# Getting started with SQLAlchemy

This lab practices SQLAlchemy on a SQLite database. SQLite is built into Python (it's the `sqlite3` module): the database is a plain file, `lab_sql.db`, with no server to install. The same code works with PostgreSQL or MySQL: only the engine's URL changes.

The exercises build on each other: each solution defines what the next ones use. If you're stuck, run the solution before moving on.

## Connection

### Creating an engine

Create an engine `engine` that connects to the SQLite database of the `lab_sql.db` file, with the debug output enabled: it shows every SQL query SQLAlchemy sends to the database, which will let you watch what follows.

In [ ]:
# Your code here

#### Additional hints

A SQLite database's URL is `sqlite:///` followed by the file's path (three slashes for a relative path). The file is created on the first connection.

```python
from sqlalchemy import create_engine

engine = create_engine(..., echo=...)
```

#### Solution

In [ ]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///lab_sql.db", echo=True)

### Running a textual query

Run the query `SELECT :a + :b`, built with `text`, passing the parameters `a=2` and `b=3`, and print its single result with `scalar_one()`. Do it twice: in a `with engine.connect() as conn:` block, then in a `with engine.begin() as conn:` block.

In the debug output, which statement ends each of the two blocks? Which of the two forms should you use to write to the database?

**Useful documentation**

- [Textual query parameters](https://docs.sqlalchemy.org/en/21/tutorial/dbapi_transactions.html#sending-parameters)
- [`Result.scalar_one`](https://docs.sqlalchemy.org/en/21/core/connections.html#sqlalchemy.engine.Result.scalar_one)

In [ ]:
# Your code here

#### Additional hints

`conn.execute` takes the query, then a dictionary of parameters (`{"a": 2, "b": 3}`). The expected result is `5`.

```python
from sqlalchemy import text

query = text("SELECT :a + :b")

with engine.connect() as conn:
  result = conn.execute(query, ...)
  print(...)

with engine.begin() as conn:
  ...
```

#### Solution

In [ ]:
from sqlalchemy import text

query = text("SELECT :a + :b")

with engine.connect() as conn:
  print(conn.execute(query, {"a": 2, "b": 3}).scalar_one())

with engine.begin() as conn:
  print(conn.execute(query, {"a": 2, "b": 3}).scalar_one())

# The connect() block ends with a ROLLBACK: whatever conn.commit() didn't commit
# is discarded. The begin() block ends with a COMMIT, or with a ROLLBACK if an
# exception leaves it. To write to the database, begin() thus commits the whole
# block, or nothing.

## SQLAlchemy Core

### Creating tables

With a `MetaData` object named `metadata`, describe two tables (`Table` objects):

- `members` (variable `table_members`):
    - `id`, integer primary key;
    - `name`, string of at most 100 characters, required;
- `addresses` (variable `table_addresses`):
    - `id`, integer primary key;
    - `street`, `city`, `zip_code` and `country`, required strings;
    - `id_member`, required, foreign key to the `id` column of `members`.

Then create these tables in the database, after dropping those a previous run may have left.

**Useful documentation**

- [Tables and columns](https://docs.sqlalchemy.org/en/21/tutorial/metadata.html#setting-up-metadata-with-table-objects)
- [Constraints](https://docs.sqlalchemy.org/en/21/tutorial/metadata.html#declaring-simple-constraints)

In [ ]:
# Your code here

#### Additional hints

"Required" is `nullable=False`. A column holding a `ForeignKey("members.id")` takes its type: no need to repeat it. `metadata.drop_all(engine)` then `metadata.create_all(engine)` drop then create every table described, and the debug output shows the `CREATE TABLE` statements.

```python
from sqlalchemy import Column, ForeignKey, Integer, MetaData, String, Table

metadata = MetaData()

table_members = Table(
  "members",
  metadata,
  Column("id", Integer, primary_key=True),
  Column("name", String(100), nullable=False),
)

table_addresses = Table(
  "addresses",
  metadata,
  ...
)

metadata.drop_all(engine)
metadata.create_all(engine)
```

#### Solution

In [ ]:
from sqlalchemy import Column, ForeignKey, Integer, MetaData, String, Table

metadata = MetaData()

table_members = Table(
  "members",
  metadata,
  Column("id", Integer, primary_key=True),
  Column("name", String(100), nullable=False),
)

table_addresses = Table(
  "addresses",
  metadata,
  Column("id", Integer, primary_key=True),
  Column("street", String, nullable=False),
  Column("city", String, nullable=False),
  Column("zip_code", String, nullable=False),
  Column("country", String, nullable=False),
  Column("id_member", ForeignKey("members.id"), nullable=False),
)

metadata.drop_all(engine)
metadata.create_all(engine)

### Inserting data

The `members` dictionary maps each member's name to the list of their addresses. Insert them into the database with `insert` statements, in a single transaction: each member, then their addresses, which need the `id` the database just gave the member. Then print the contents of both tables.

**Useful documentation**

- [`CursorResult.inserted_primary_key`](https://docs.sqlalchemy.org/en/21/core/connections.html#sqlalchemy.engine.CursorResult.inserted_primary_key)

In [ ]:
members = {
  "Jean Dupont": [
    {
      "street": "1 rue Georges Clemenceau",
      "city": "Nantes",
      "zip_code": "44000",
      "country": "France",
    },
    {
      "street": "26 boulevard de la Prairie au Duc",
      "city": "Nantes",
      "zip_code": "44200",
      "country": "France",
    },
  ],
  "John Doe": [
    {
      "street": "Quai Ferdinand Favre",
      "city": "Nantes",
      "zip_code": "44000",
      "country": "France",
    },
    {
      "street": "Dyrehaven",
      "city": "Klampenborg",
      "zip_code": "2930",
      "country": "Denmark",
    },
  ],
  "Sherlock Holmes": [
    {
      "street": "221b Baker Street",
      "city": "London",
      "zip_code": "NW1 6XE",
      "country": "United Kingdom",
    },
  ],
}

In [ ]:
# Your code here

#### Additional hints

`conn.execute(insert(table_members).values(name=name))` inserts a row, and `result.inserted_primary_key[0]` is the `id` it was given. `conn.execute(insert(table_addresses), [...])` inserts several rows at once, one per dictionary of the list, and `dict(address, id_member=id_member)` completes an address dictionary. You should get 3 members and 5 addresses.

```python
from sqlalchemy import insert, select

with engine.begin() as conn:
  for name, addresses in members.items():
    result = conn.execute(...)
    id_member = ...
    conn.execute(
      insert(table_addresses),
      [dict(address, id_member=id_member) for address in addresses],
    )

with engine.connect() as conn:
  for row in conn.execute(select(table_members)):
    print(row)
  ...
```

#### Solution

In [ ]:
from sqlalchemy import insert, select

with engine.begin() as conn:
  for name, addresses in members.items():
    result = conn.execute(insert(table_members).values(name=name))
    id_member = result.inserted_primary_key[0]
    conn.execute(
      insert(table_addresses),
      [dict(address, id_member=id_member) for address in addresses],
    )

with engine.connect() as conn:
  for row in conn.execute(select(table_members)):
    print(row)
  for row in conn.execute(select(table_addresses)):
    print(row)

### Rolling back a transaction

In a `with engine.begin() as conn:` block, insert a member `Arsène Lupin`, then an address of this member without a `country`: the database refuses it, since `country` is required. Catch the `IntegrityError` exception outside the block, then count the members named `Arsène Lupin`. What became of the member inserted before the error?

**Useful documentation**

- [`IntegrityError`](https://docs.sqlalchemy.org/en/21/core/exceptions.html#sqlalchemy.exc.IntegrityError)

In [ ]:
# Your code here

#### Additional hints

`IntegrityError` is imported from `sqlalchemy.exc`. When an exception leaves a `begin()` block, SQLAlchemy rolls the transaction back (`ROLLBACK` in the debug output) instead of committing it.

```python
from sqlalchemy.exc import IntegrityError

try:
  with engine.begin() as conn:
    result = conn.execute(insert(table_members).values(name="Arsène Lupin"))
    conn.execute(insert(table_addresses).values(...))
except IntegrityError as error:
  print("Refused :", error.orig)

with engine.connect() as conn:
  lupins = conn.scalars(select(table_members.c.id).where(...)).all()
  print(len(lupins))
```

#### Solution

In [ ]:
from sqlalchemy.exc import IntegrityError

try:
  with engine.begin() as conn:
    result = conn.execute(insert(table_members).values(name="Arsène Lupin"))
    conn.execute(
      insert(table_addresses).values(
        street="36 rue Raynouard",
        city="Paris",
        zip_code="75016",
        id_member=result.inserted_primary_key[0],
      )
    )
except IntegrityError as error:
  print("Refused:", error.orig)

with engine.connect() as conn:
  lupins = conn.scalars(
    select(table_members.c.id).where(table_members.c.name == "Arsène Lupin")
  ).all()
  print(len(lupins))

# 0: the refused address made an exception leave the begin() block, which thus
# rolled back the whole transaction, the member's insertion included.

### Selecting data

Print, in alphabetical order, the names of the members who have at least one address in France, each one once.

**Useful documentation**

- [Joins](https://docs.sqlalchemy.org/en/21/tutorial/data_select.html#explicit-from-clauses-and-joins)
- [`Select.distinct`](https://docs.sqlalchemy.org/en/21/core/selectable.html#sqlalchemy.sql.expression.Select.distinct)

In [ ]:
# Your code here

#### Additional hints

`select(table_members.c.name).join(table_addresses)` joins the two tables: SQLAlchemy finds the join condition from the foreign key. A member appears once per address in France, hence `distinct()`. `conn.scalars(...)` returns each row's first column. You should get `['Jean Dupont', 'John Doe']`.

```python
with engine.connect() as conn:
  names = conn.scalars(
    select(table_members.c.name)
    .join(...)
    .where(...)
    .distinct()
    .order_by(table_members.c.name)
  ).all()
  print(names)
```

#### Solution

In [ ]:
with engine.connect() as conn:
  names = conn.scalars(
    select(table_members.c.name)
    .join(table_addresses)
    .where(table_addresses.c.country == "France")
    .distinct()
    .order_by(table_members.c.name)
  ).all()
  print(names)

### Updating data

With an `update` statement, rename `John Doe` to `Jane Doe`. Print the `members` table before and after the update.

**Useful documentation**

- [`update`](https://docs.sqlalchemy.org/en/21/tutorial/data_update.html#the-update-sql-expression-construct)

In [ ]:
# Your code here

#### Additional hints

`update(table).where(condition).values(column=value)`: run it in a `begin()` block, so that it's committed.

```python
from sqlalchemy import update

with engine.connect() as conn:
  print(conn.execute(select(table_members)).all())

with engine.begin() as conn:
  conn.execute(update(table_members).where(...).values(...))

with engine.connect() as conn:
  print(...)
```

#### Solution

In [ ]:
from sqlalchemy import update

with engine.connect() as conn:
  print(conn.execute(select(table_members)).all())

with engine.begin() as conn:
  conn.execute(
    update(table_members)
    .where(table_members.c.name == "John Doe")
    .values(name="Jane Doe")
  )

with engine.connect() as conn:
  print(conn.execute(select(table_members)).all())

### Deleting data

With `delete` statements, delete `Jane Doe` in a single transaction, then print the number of addresses left. Delete her addresses first: otherwise, they would refer to a member who no longer exists. PostgreSQL would then refuse to delete the member; SQLite only checks foreign keys when asked to.

**Useful documentation**

- [`delete`](https://docs.sqlalchemy.org/en/21/tutorial/data_update.html#the-delete-sql-expression-construct)
- [`Select.scalar_subquery`](https://docs.sqlalchemy.org/en/21/core/selectable.html#sqlalchemy.sql.expression.Select.scalar_subquery)

In [ ]:
# Your code here

#### Additional hints

`select(table_members.c.id).where(...)` gives Jane Doe's `id`, and `.scalar_subquery()` makes it a subquery that can be compared to a column in the condition of the addresses' `delete`. There should be 3 addresses left.

```python
from sqlalchemy import delete

id_jane = select(table_members.c.id).where(...).scalar_subquery()

with engine.begin() as conn:
  conn.execute(delete(table_addresses).where(table_addresses.c.id_member == id_jane))
  conn.execute(delete(table_members).where(...))

with engine.connect() as conn:
  print(len(conn.execute(select(table_addresses)).all()))
```

#### Solution

In [ ]:
from sqlalchemy import delete

id_jane = (
  select(table_members.c.id).where(table_members.c.name == "Jane Doe").scalar_subquery()
)

with engine.begin() as conn:
  conn.execute(delete(table_addresses).where(table_addresses.c.id_member == id_jane))
  conn.execute(delete(table_members).where(table_members.c.name == "Jane Doe"))

with engine.connect() as conn:
  print(len(conn.execute(select(table_addresses)).all()))

## SQLAlchemy ORM

Now the same data with the ORM: Python classes describe the tables, and a session keeps their objects in sync with the database. The `members` and `addresses` tables are created anew.

### Describing the models

Describe the two tables as classes, in the typed style: a `Base` class that inherits from `DeclarativeBase`, then:

- `Member` (table `members`): `id`, integer primary key; `name`, string of at most 100 characters; `addresses`, the list of their addresses;
- `Address` (table `addresses`): `id`, integer primary key; `street`, `city`, `zip_code` and `country`, strings; `id_member`, foreign key to `members.id`; `member`, the member it belongs to.

Every column is required. `Member.addresses` and `Address.member` are the two sides of one one-to-many relationship. An address doesn't exist without its member: declare on `Member.addresses` the `"all, delete-orphan"` cascade, which deletes the addresses of a deleted member, or those removed from their list.

Drop the existing tables, then create these ones.

**Useful documentation**

- [One-to-many relationship](https://docs.sqlalchemy.org/en/21/orm/basic_relationships.html#one-to-many)
- [`delete-orphan` cascade](https://docs.sqlalchemy.org/en/21/orm/cascades.html#delete-orphan)

In [ ]:
# Your code here

#### Additional hints

With `Mapped[...]`, the annotation gives the type and whether it's required: `Mapped[str]` is a required column (`Mapped[str | None]` would make it optional), and `mapped_column` is only for the rest (primary key, length, foreign key). Each side of the relationship names the other with `back_populates`. The debug output should show two `CREATE TABLE` statements, one of them with a `FOREIGN KEY`.

```python
from sqlalchemy import ForeignKey, String
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship


class Base(DeclarativeBase):
  pass


class Member(Base):
  __tablename__ = "members"
  id: Mapped[int] = mapped_column(primary_key=True)
  name: Mapped[str] = mapped_column(String(100))
  addresses: Mapped[list["Address"]] = relationship(
    back_populates=..., cascade=...
  )


class Address(Base):
  __tablename__ = "addresses"
  id: Mapped[int] = mapped_column(primary_key=True)
  street: Mapped[str]
  ...
  id_member: Mapped[int] = mapped_column(...)
  member: Mapped[Member] = relationship(...)


Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)
```

#### Solution

In [ ]:
from sqlalchemy import ForeignKey, String
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship


class Base(DeclarativeBase):
  pass


class Member(Base):
  __tablename__ = "members"
  id: Mapped[int] = mapped_column(primary_key=True)
  name: Mapped[str] = mapped_column(String(100))
  addresses: Mapped[list["Address"]] = relationship(
    back_populates="member", cascade="all, delete-orphan"
  )


class Address(Base):
  __tablename__ = "addresses"
  id: Mapped[int] = mapped_column(primary_key=True)
  street: Mapped[str]
  city: Mapped[str]
  zip_code: Mapped[str]
  country: Mapped[str]
  id_member: Mapped[int] = mapped_column(ForeignKey("members.id"))
  member: Mapped[Member] = relationship(back_populates="addresses")


Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

### Inserting objects

With a `Session`, insert the members of the `members` dictionary and their addresses: create `Member` and `Address` objects, add them to the session, then commit. The debug output shows the `INSERT` statements the session sends on `commit()`.

In [ ]:
# Your code here

#### Additional hints

Adding the members is enough: their addresses, passed as `addresses=[...]`, follow through the relationship. `Address(**address)` creates an address from a dictionary of `members`.

```python
from sqlalchemy.orm import Session

with Session(engine) as session:
  for name, addresses in members.items():
    member = Member(name=name, addresses=[...])
    session.add(member)
  session.commit()
```

#### Solution

In [ ]:
from sqlalchemy.orm import Session

with Session(engine) as session:
  for name, addresses in members.items():
    member = Member(name=name, addresses=[Address(**address) for address in addresses])
    session.add(member)
  session.commit()

### Selecting objects

Print, in alphabetical order, the names of the members who have at least one address in France, with a `select` query on `Member` run by a session.

In [ ]:
# Your code here

#### Additional hints

`select(Member).join(Member.addresses)` joins through the relationship, `.where(Address.country == "France")` filters, `.distinct()` avoids duplicates and `.order_by(Member.name)` sorts. `session.scalars(...)` returns `Member` objects. You should get Jean Dupont and John Doe.

```python
with Session(engine) as session:
  query = select(Member).join(...).where(...).distinct()
  for member in session.scalars(query):
    print(member.name)
```

#### Solution

In [ ]:
with Session(engine) as session:
  query = (
    select(Member)
    .join(Member.addresses)
    .where(Address.country == "France")
    .distinct()
    .order_by(Member.name)
  )
  for member in session.scalars(query):
    print(member.name)

### Following a relationship

Print each member, then the city and the country of each of their addresses, going through `member.addresses`. How many `SELECT` statements does the debug output show? Do it again with `.options(selectinload(Member.addresses))` added to the query: how many `SELECT` statements now?

**Useful documentation**

- [Loader strategies](https://docs.sqlalchemy.org/en/21/tutorial/orm_related_objects.html#loader-strategies)
- [`selectinload`](https://docs.sqlalchemy.org/en/21/orm/queryguide/relationships.html#sqlalchemy.orm.selectinload)

In [ ]:
# Your code here

#### Additional hints

By default, a relationship is loaded on demand (lazy loading): the first access to `member.addresses` sends a query for this member alone. `selectinload` loads the addresses of every member in a single extra query.

```python
from sqlalchemy.orm import selectinload

with Session(engine) as session:
  for member in session.scalars(select(Member)):
    print(member.name)
    for address in member.addresses:
      print("  ", address.city, address.country)

with Session(engine) as session:
  query = select(Member).options(...)
  ...
```

#### Solution

In [ ]:
from sqlalchemy.orm import selectinload

with Session(engine) as session:
  for member in session.scalars(select(Member)):
    print(member.name)
    for address in member.addresses:
      print("  ", address.city, address.country)

print("*" * 50)

with Session(engine) as session:
  query = select(Member).options(selectinload(Member.addresses))
  for member in session.scalars(query):
    print(member.name)
    for address in member.addresses:
      print("  ", address.city, address.country)

# Without an option: 4 SELECTs, one for the members, then one per member on the
# first access to member.addresses (lazy loading). With selectinload: 2 SELECTs,
# one for the members, one for the addresses of every member (WHERE ... IN).
# Over thousands of members, the gap matters: it's the "N + 1" problem.

### Modifying an object

Fetch the member `John Doe` with a `select` query, rename him `Jane Doe` by changing his `name` attribute, then commit. Print the list of the members' names before and after.

**Useful documentation**

- [Updating an object](https://docs.sqlalchemy.org/en/21/tutorial/orm_data_manipulation.html#updating-orm-objects-using-the-unit-of-work-pattern)

In [ ]:
# Your code here

#### Additional hints

`session.scalars(query).one()` returns the single object found, and raises an exception if there are none or several. On `commit()`, the session sends the `UPDATE` itself: the debug output shows it.

```python
with Session(engine) as session:
  print(session.scalars(select(Member.name)).all())
  john = session.scalars(select(Member).where(...)).one()
  john.name = ...
  session.commit()
  print(...)
```

#### Solution

In [ ]:
with Session(engine) as session:
  print(session.scalars(select(Member.name)).all())
  john = session.scalars(select(Member).where(Member.name == "John Doe")).one()
  john.name = "Jane Doe"
  session.commit()
  print(session.scalars(select(Member.name)).all())

### Deleting an object

Fetch `Jane Doe`, delete her with the session, commit, then print the number of addresses left in the database. What happened to her addresses? What would have happened without the `"all, delete-orphan"` cascade of `Member.addresses`?

**Useful documentation**

- [Deleting an object](https://docs.sqlalchemy.org/en/21/tutorial/orm_data_manipulation.html#deleting-orm-objects-using-the-unit-of-work-pattern)

In [ ]:
# Your code here

#### Additional hints

`session.delete(obj)` marks the object for deletion, and `commit()` sends the `DELETE` statements: look for them in the debug output. There should be 3 addresses left.

```python
with Session(engine) as session:
  jane = session.scalars(select(Member).where(...)).one()
  session.delete(...)
  session.commit()
  print(len(session.scalars(select(Address)).all()))
```

#### Solution

In [ ]:
with Session(engine) as session:
  jane = session.scalars(select(Member).where(Member.name == "Jane Doe")).one()
  session.delete(jane)
  session.commit()
  print(len(session.scalars(select(Address)).all()))

# 3 addresses: the "all, delete-orphan" cascade deleted Jane Doe's two addresses
# along with her (a DELETE of the addresses, then the member's). Without a
# cascade, SQLAlchemy would have set their id_member to NULL (an UPDATE), which
# the required column refuses: commit() would have raised an IntegrityError.